# BoGrad Ablation (M1) — Colab Launcher

Runs all 9 BoGrad ablation axes (10.01–10.09) on a Colab GPU, writing results to
Google Drive so they survive disconnects. **Idempotent**: re-running resumes —
the per-cell `JobManager` skips any cell whose `results.json` is already complete.

Recommended runtime: **GPU** (T4 is fine; A100/L4 faster). Run the cells top to
bottom. If Colab drops, just re-open and run all cells again — it picks up where
it left off.

## 1. Clone the repo (or pull if already cloned)

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rayden96/MastersDissertationExperiments.git"
BRANCH   = "m0-infrastructure"   # change to your working branch
REPO_DIR = "/content/MastersDissertationExperiments"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
print("repo at", REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

## 2. Install dependencies
torch/torchvision/sklearn are preinstalled on Colab; `datasets` (HuggingFace, for the text dataset in later phases) usually is not.

In [ ]:
import importlib, subprocess, sys
for pkg, pip_name in [("torch", None), ("torchvision", None), ("sklearn", "scikit-learn"), ("datasets", "datasets")]:
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: ok")
    except ImportError:
        name = pip_name or pkg
        print(f"installing {name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", name], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))

## 3. Mount Drive and point results there
`common.storage.get_results_root()` honours `DISSERTATION_RESULTS_ROOT`. We set it
to a Drive folder so every `results/` write lands on Drive and survives restarts.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
RESULTS_ROOT = "/content/drive/MyDrive/dissertation/results"
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.environ["DISSERTATION_RESULTS_ROOT"] = RESULTS_ROOT
print("results ->", RESULTS_ROOT)
# The code writes to get_results_root() (= this dir). All axis runs AND the
# master table land under {RESULTS_ROOT}/10_bograd_ablation/... and persist
# automatically — no symlinks needed.

## 4. Sanity check the imports (fast, no training)

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "PaperReadyExperiments"))
from common.methods import build_method, METHODS, BASE_OPTIMIZERS
from common.optimizers import BoGrad, COSGD, GradDrop, SignSGD
print("methods:", METHODS)
print("bases  :", BASE_OPTIMIZERS)
print("imports OK — ready to sweep")

## 5. Smoke test (optional, ~2 min) — prove the pipeline before the long run

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/10_bograd_ablation
!python run_all.py --smoke --axes 01 04

## 6. The full M1 sweep
Proxy schedule (short epochs) — enough to rank settings and move the interference
metrics; final tuned numbers come later in the bakeoff. Adjust `--epochs`,
`--bases`, `--seeds`, or `--axes` to fit your Colab session. Re-run any time to
resume.

Rough budget: ~100 ms/step on CIFAR-10 small CNN with measurement; a full axis
(≈100 cells × ~3.5k steps) is a few hours. Run axes in batches across sessions if
needed — resume is automatic.

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/10_bograd_ablation
# Example: the cheaper axes first (01, 03, 06) on all 4 bases, 10-epoch proxy.
!python run_all.py --axes 01 03 06 --epochs 10 --seeds 2026 2027 2028

In [ ]:
# The rest (04, 05, 07, 08) + LR retune (02). Run in a later session if needed.
%cd {REPO_DIR}/PaperReadyExperiments/10_bograd_ablation
!python run_all.py --axes 02 04 05 07 08 --epochs 10 --seeds 2026 2027 2028

## 7. Build the master "when / what / why" table (10.09)

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/10_bograd_ablation
!python 09_cross_summary/run.py
# Per-axis figures:
!python 01_buffer_K/plot.py || true

## 8. (Optional) commit results-summaries back
Raw `results/` are gitignored and live on Drive. If you want the small
`summary.json` / `master_table.json` in git, copy them out and commit from a
machine with push credentials — Colab clones are read-only by default.